# 02 — BM25 baseline & retrieval eval

**The question this notebook answers:** with 4,133 chunks and 12 graded queries, how often does
BM25 put the right *document* in the top 10?

## Eval-first

The value of this project rests on one rule: **every pipeline change is measured against the same
qrels**. So the first thing built is not a retriever — it is the ruler. BM25, dense and hybrid
all get scored by the code below, which is why the comparison in the README is a comparison
between things that were actually measured the same way.

## Conventions fixed up front

| Concept | Definition |
|---|---|
| "relevant" (binary) | `relevance >= 1` — used by Recall@k, Precision@k, MRR |
| graded relevance | integer `0 / 1 / 2` — used by nDCG@k |
| ranking unit | the **document** (`doc_id`), not the chunk — qrels are graded per document |

---
# Part A — the metric functions


In [1]:
# ── Setup: chunks + qrels (run once) ───────────────────────────
import json, math, random
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS = ROOT / "data" / "processed" / "chunks.jsonl"
QRELS = ROOT / "eval" / "qrels.csv"

chunks = pd.read_json(CHUNKS, lines=True)
chunks["doc_id"] = chunks["doc_id"].astype(str)

qrels = pd.read_csv(QRELS, dtype={"doc_id": str})
qrels = qrels.dropna(subset=["doc_id", "relevance"])

QUERIES = dict(zip(qrels["query_id"], qrels["query"]))            # {qid: question}
QUERY_CLASS = dict(zip(qrels["query_id"], qrels["query_class"]))  # {qid: literal|paraphrase|multi-doc}
GOLD_BINARY = qrels[qrels["relevance"] >= 1].groupby("query_id")["doc_id"].apply(set).to_dict()
GOLD_GRADED = qrels.groupby("query_id").apply(
    lambda g: dict(zip(g["doc_id"], g["relevance"])), include_groups=False).to_dict()

print(f"chunks : {len(chunks):,} · {chunks['doc_id'].nunique()} documents")
print(f"queries: {len(QUERIES)} · qrels rows: {len(qrels)}")
print(f"classes: {pd.Series(QUERY_CLASS).value_counts().to_dict()}")


chunks : 4,133 · 179 documents
queries: 12 · qrels rows: 31
classes: {'literal': 4, 'paraphrase': 4, 'multi-doc': 4}


In [2]:
# ── The four metric functions ─────────────────────────────────
# Convention: `ranked_docs` is a list of doc_ids, best first. A query with no
# relevant result scores 0.0 — not None, and never an exception, so that a
# per-query breakdown stays a table instead of a traceback.

def recall_at_k(ranked_docs, gold, k):
    """Share of the gold documents found in the top k.
    (gold documents inside the top k) / (all gold documents)
    """
    if len(gold) == 0:
        return 0.0

    count = 0
    for doc in ranked_docs[:k]:
        if doc in gold:
            count += 1
    return count / len(gold)

def precision_at_k(ranked_docs, gold, k):
    """Share of the top k positions that are gold documents.
    Note the denominator is k, not the number of gold documents: with the gold
    sets in this project (1-2 documents) precision@10 tops out structurally at
    0.1-0.2, which is why MRR and nDCG carry the ranking story.
    """
    if len(gold) == 0:
        return 0.0
    count = 0
    for doc in ranked_docs[:k]:
        if doc in gold:
            count += 1
    return count / k

def mrr(ranked_docs, gold):
    """Reciprocal rank of the FIRST gold document for one query.
    Rank 1 -> 1.0 · rank 2 -> 0.5 · rank 5 -> 0.2 · no gold found -> 0.0
    """
    for i, doc in enumerate(ranked_docs):
        if doc in gold:
            return 1 / (i + 1)

    return 0.0

def ndcg_at_k(ranked_docs, gold_graded, k):
    """nDCG@k with graded relevance (0/1/2).
    DCG@k  = sum over positions i (1-based): relevance / log2(i + 1)
    IDCG@k = the same sum for the IDEAL ordering (gold sorted by relevance)
    nDCG   = DCG / IDCG   (0.0 when IDCG is 0)
    Dividing by IDCG puts every query on a comparable [0, 1] scale, which is what
    makes averaging across queries meaningful.
    """
    DCG = 0.0
    IDCG = 0.0

    for i, doc in enumerate(ranked_docs[:k], start=1):
        rel = gold_graded.get(doc, 0)
        DCG += rel / (math.log2(i + 1))

    for i, rel in enumerate(sorted(gold_graded.values(), reverse=True)[:k], start=1):
        IDCG += rel / (math.log2(i + 1))

    if IDCG == 0:
        return 0.0
    return DCG / IDCG


## Self-test before trusting any of it

Hand-built case: ranking `[C, A, B]` (C is irrelevant), gold `{A: 2, B: 1}`, k = 3.

- DCG@3 = 2/log2(2) + 1/log2(3) = 2.0000 + 0.6309 = 2.6309
- IDCG@3 = 2/log2(2) + 1/log2(3) = 2.6309 (the ideal ordering is already [2, 1])
- nDCG@3 = 0.6697

The cell does not touch the functions — it compares their output against these hand-computed
values and prints ❌ on any mismatch.


In [3]:
# ── Self-test ─────────────────────────────────────────────────
RANKED = ["C", "A", "B"]
GOLD_BIN = {"A", "B"}
GOLD_GRD = {"A": 2, "B": 1}
K = 3

EXPECTED = {"Recall@3": 1.0000, "Precision@3": 0.6667, "MRR": 0.5000, "nDCG@3": 0.6697}

got = {}
try:
    got["Recall@3"] = recall_at_k(RANKED, GOLD_BIN, K)
    got["Precision@3"] = precision_at_k(RANKED, GOLD_BIN, K)
    got["MRR"] = mrr(RANKED, GOLD_BIN)
    got["nDCG@3"] = ndcg_at_k(RANKED, GOLD_GRD, K)
except Exception as e:
    print(f"a function is incomplete: {type(e).__name__}: {e}")

for name, want in EXPECTED.items():
    if name not in got:
        print(f"{name:12s} did not run")
        continue
    v = got[name]
    print(f"{'OK ' if abs(v - want) < 0.001 else 'BAD'} {name:12s} = {v:.4f}   |   expected = {want:.4f}")

# edge cases: no relevant result at all
print("\nedge cases — no gold document retrieved:")
print(f"   mrr(['X','Y'], {{'A'}})             = {mrr(['X','Y'], {'A'})}   (expected 0.0)")
print(f"   recall_at_k(['X','Y'], {{'A'}}, 2)  = {recall_at_k(['X','Y'], {'A'}, 2)}   (expected 0.0)")
print(f"   precision_at_k([], {{'A'}}, 2)      = {precision_at_k([], {'A'}, 2)}   (expected 0.0)")


OK  Recall@3     = 1.0000   |   expected = 1.0000
OK  Precision@3  = 0.6667   |   expected = 0.6667
OK  MRR          = 0.5000   |   expected = 0.5000
OK  nDCG@3       = 0.6697   |   expected = 0.6697

edge cases — no gold document retrieved:
   mrr(['X','Y'], {'A'})             = 0.0   (expected 0.0)
   recall_at_k(['X','Y'], {'A'}, 2)  = 0.0   (expected 0.0)
   precision_at_k([], {'A'}, 2)      = 0.0   (expected 0.0)


# Part B — BM25

In [4]:
# ── Setup ─────────────────────────────────────────────────────
import sys
import numpy as np
from rank_bm25 import BM25Okapi

sys.path.insert(0, str(ROOT / "scripts"))
from explore import tokenize          # the same tokenizer the search tool uses

TEXTS = chunks["text"].tolist()        # 4,133 chunks
print(f"{len(TEXTS):,} chunks ready")
print("  first chunk, tokenised:", tokenize(TEXTS[0])[:12])


4,133 chunks ready
  first chunk, tokenised: ['overviews', 'tinh', 'nang', 'tich', 'hop', 'tri', 'tue', 'nhan', 'tao', 'google', 'tim', 'kiem']


In [5]:
# ── Index + retrieve ──────────────────────────────────────────

def build_index(texts):
    """Build the BM25 index over the chunk texts."""
    corpus_tokens = [tokenize(text) for text in texts]
    return BM25Okapi(corpus_tokens)


def retrieve(query, k=10):
    """Top-k CHUNKS, best first.

    Output contract, relied on by the sanity check and by docs_from_chunks:
      list of {"chunk_id": str, "doc_id": str, "score": float, "text": str}
    """
    q_tokens = tokenize(query)
    scores = BM25.get_scores(q_tokens)
    top_k_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        results.append({
            "chunk_id": chunks.iloc[idx]["chunk_id"],
            "doc_id": chunks.iloc[idx]["doc_id"],
            "score": scores[idx],
            "text": chunks.iloc[idx]["text"]
        })

    return results


def docs_from_chunks(hits, k=10):
    """Collapse a ranked CHUNK list into a ranked DOCUMENT list.

    A document's score is the MAXIMUM of its chunk scores. One passage containing
    the answer is enough to make the document relevant, so summing would reward
    long documents for being long, and averaging would punish a precise answer
    for sitting in a document that also drifts off topic.

    Each doc_id appears exactly once, sorted best first, trimmed to k.
    """
    doc_scores = {}
    for hit in hits:
        doc_id = hit["doc_id"]
        score = hit["score"]
        if doc_id not in doc_scores or score > doc_scores[doc_id]:
            doc_scores[doc_id] = score

    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in sorted_docs[:k]]


In [6]:
# ── Sanity check before running the full eval ─────────────────
BM25 = build_index(TEXTS)

CHECKS = [
    ("q001", {"24345", "19893713"}),
    ("q002", {"30712", "20023680"}),
    ("q005", {"16665860", "24278"}),
]

for qid, gold in CHECKS:
    hits = retrieve(QUERIES[qid], k=10)
    docs = docs_from_chunks(hits, k=5)
    found = gold & set(docs)
    print(f"\n=== {qid} · class {QUERY_CLASS[qid]} ===")
    print(f"  query       : {QUERIES[qid][:70]}")
    print(f"  gold        : {sorted(gold)}")
    print(f"  top-3 chunk : {[(h['doc_id'], round(float(h['score']), 1)) for h in hits[:3]]}")
    print(f"  top-5 docs  : {docs}")
    print(f"  -> {len(found)}/{len(gold)} gold documents found in the top 5")



=== q001 · class literal ===
  query       : Học không có giám sát là gì?
  gold        : ['19893713', '24345']
  top-3 chunk : [('19895408', 8.2), ('24345', 8.2), ('19893713', 8.0)]
  top-5 docs  : ['19895408', '24345', '19893713', '3339820', '16760383']
  -> 2/2 gold documents found in the top 5

=== q002 · class literal ===
  query       : Phân loại nhị phân là gì?
  gold        : ['20023680', '30712']
  top-3 chunk : [('20023680', 9.5), ('20023680', 9.4), ('30712', 9.2)]
  top-5 docs  : ['20023680', '30712', '19991608', '20015375', '667810']
  -> 2/2 gold documents found in the top 5

=== q005 · class paraphrase ===
  query       : Vì sao mô hình đạt kết quả rất tốt trên dữ liệu huấn luyện nhưng lại k
  gold        : ['16665860', '24278']
  top-3 chunk : [('24278', 25.8), ('20023678', 24.5), ('20002585', 24.3)]
  top-5 docs  : ['24278', '20023678', '20002585', '14920124', '20022788']
  -> 1/2 gold documents found in the top 5


# Part C — random baseline + full BM25 eval

In [7]:
# ── Ranking adapters + the eval loop ──────────────────────────

def bm25_rank(query, k=10, n_chunks=50):
    """Adapter: BM25 chunks -> ranked documents, for the metric functions.

    Retrieval depth matters here. Aggregating only the top 10 chunks yields 6-8
    distinct documents, so rank 10 is never even reachable and the metric would
    understate the retriever for purely structural reasons. Taking 50 chunks and
    then collapsing to k documents is what makes the k=10 numbers mean what they
    say.
    """
    hits = retrieve(query, k=n_chunks)
    return docs_from_chunks(hits, k)


def evaluate(rank_fn, k=10):
    """Score one ranking function over every query -> one DataFrame row per query."""
    rows = []
    for qid, q in QUERIES.items():
        ranked = rank_fn(q, k=k)
        rows.append({
            'query_id': qid,
            'query_class': QUERY_CLASS[qid],
            'recall': recall_at_k(ranked, GOLD_BINARY[qid], k),
            'precision': precision_at_k(ranked, GOLD_BINARY[qid], k),
            'mrr': mrr(ranked, GOLD_BINARY[qid]),
            'ndcg': ndcg_at_k(ranked, GOLD_GRADED[qid], k),
        })

    return pd.DataFrame(rows)


In [8]:
# ── Reporting ─────────────────────────────────────────────────
COLS = ["recall", "precision", "mrr", "ndcg"]

def report(df, name, k=10):
    print(f"\n{'='*70}\n{name}   (k={k})\n{'='*70}")
    print("\n-- mean by query class --")
    print(df.groupby("query_class")[COLS].mean().round(3).to_string())
    print("\n-- overall --")
    print(df[COLS].mean().round(3).to_string())
    print("\n-- per query --")
    print(df[["query_id", "query_class"] + COLS].round(3).to_string(index=False))

RESULTS_BM25 = evaluate(bm25_rank, k=10)
report(RESULTS_BM25, "BM25")



BM25   (k=10)

-- mean by query class --
             recall  precision    mrr   ndcg
query_class                                 
literal       1.000      0.150  0.875  0.882
multi-doc     1.000      0.275  0.750  0.758
paraphrase    0.625      0.125  0.528  0.398

-- overall --
recall       0.875
precision    0.183
mrr          0.718
ndcg         0.680

-- per query --
query_id query_class  recall  precision   mrr  ndcg
    q001     literal     1.0        0.2 0.500 0.670
    q002     literal     1.0        0.2 1.000 0.860
    q003     literal     1.0        0.1 1.000 1.000
    q004     literal     1.0        0.1 1.000 1.000
    q005  paraphrase     0.5        0.1 1.000 0.380
    q006  paraphrase     0.0        0.0 0.000 0.000
    q007  paraphrase     1.0        0.2 0.111 0.339
    q008  paraphrase     1.0        0.2 1.000 0.875
    q009   multi-doc     1.0        0.3 0.500 0.648
    q010   multi-doc     1.0        0.2 1.000 0.832
    q011   multi-doc     1.0        0.3 1.000 0.951
 

In [9]:
# ── Random baseline ───────────────────────────────────────────
# Without a floor, an MRR of 0.72 is just a number. Random ranking shows how much
# of the score is the retriever and how much is the corpus being small.
ALL_DOC_IDS = sorted(chunks["doc_id"].unique())      # 179 documents

def random_rank(query, k=10, seed=42):
    """Rank documents at random, with a query-dependent but fixed seed.

    The seed must be reproducible, and it must depend on the query: a shared
    seed would hand every query the same ranking, which is not a baseline.
    """
    rng = random.Random(f"{seed}-{query}")
    return rng.sample(ALL_DOC_IDS, k)

RESULTS_RANDOM = evaluate(random_rank, k=10)
report(RESULTS_RANDOM, "RANDOM (baseline)")



RANDOM (baseline)   (k=10)

-- mean by query class --
             recall  precision    mrr  ndcg
query_class                                
literal       0.000      0.000  0.000  0.00
multi-doc     0.000      0.000  0.000  0.00
paraphrase    0.125      0.025  0.031  0.03

-- overall --
recall       0.042
precision    0.008
mrr          0.010
ndcg         0.010

-- per query --
query_id query_class  recall  precision   mrr  ndcg
    q001     literal     0.0        0.0 0.000  0.00
    q002     literal     0.0        0.0 0.000  0.00
    q003     literal     0.0        0.0 0.000  0.00
    q004     literal     0.0        0.0 0.000  0.00
    q005  paraphrase     0.5        0.1 0.125  0.12
    q006  paraphrase     0.0        0.0 0.000  0.00
    q007  paraphrase     0.0        0.0 0.000  0.00
    q008  paraphrase     0.0        0.0 0.000  0.00
    q009   multi-doc     0.0        0.0 0.000  0.00
    q010   multi-doc     0.0        0.0 0.000  0.00
    q011   multi-doc     0.0        0.0 0.000

# Part D — dense retrieval

In [10]:
# ── Dense retrieval over pre-computed embeddings ──────────────
import os
# Redirect the HuggingFace cache before importing sentence_transformers.
# The default (~/.cache/huggingface) is access-denied on this machine, so the
# cache lives on D:. setdefault means an externally set HF_HOME still wins —
# delete these two lines if the default cache location works for you.
os.environ.setdefault("HF_HOME", r"D:\Hermes-Workspace\hf-cache")
os.environ.setdefault("HF_HUB_CACHE", r"D:\Hermes-Workspace\hf-cache\hub")

import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
EMB = np.load(ROOT / "data" / "processed" / "embeddings_minilm.npy")
print("EMB:", EMB.shape)


def dense_rank(query, k=10, n_chunks=50):
    """Embedding retrieval — same output contract as bm25_rank.

    Both sides are L2-normalised, so the dot product is the cosine similarity:
    4133 x 384 @ 384 -> 4133 scores. Chunks are then collapsed to documents by
    the same max rule, which keeps the two retrievers comparable.
    """
    q_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = EMB @ q_vec
    top_indices = np.argsort(scores)[::-1][:n_chunks]

    hits = [
        {
            "chunk_id": chunks.iloc[idx]["chunk_id"],
            "doc_id": chunks.iloc[idx]["doc_id"],
            "score": scores[idx],
            "text": chunks.iloc[idx]["text"],
        }
        for idx in top_indices
    ]

    return docs_from_chunks(hits, k)


RESULTS_DENSE = evaluate(dense_rank, k=10)
report(RESULTS_DENSE, "DENSE — MiniLM multilingual")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

EMB: (4133, 384)



DENSE — MiniLM multilingual   (k=10)

-- mean by query class --
             recall  precision    mrr   ndcg
query_class                                 
literal       0.750      0.125  0.750  0.738
multi-doc     0.833      0.225  0.750  0.668
paraphrase    0.875      0.150  0.362  0.432

-- overall --
recall       0.819
precision    0.167
mrr          0.621
ndcg         0.613

-- per query --
query_id query_class  recall  precision  mrr  ndcg
    q001     literal   1.000        0.2 1.00 1.000
    q002     literal   1.000        0.2 1.00 0.950
    q003     literal   0.000        0.0 0.00 0.000
    q004     literal   1.000        0.1 1.00 1.000
    q005  paraphrase   0.500        0.1 0.50 0.240
    q006  paraphrase   1.000        0.1 0.25 0.431
    q007  paraphrase   1.000        0.2 0.50 0.627
    q008  paraphrase   1.000        0.2 0.20 0.429
    q009   multi-doc   0.667        0.2 1.00 0.634
    q010   multi-doc   1.000        0.2 0.50 0.693
    q011   multi-doc   1.000        0.3 1

In [11]:
COMPARE = pd.DataFrame({
    "Random": RESULTS_RANDOM[COLS].mean(),
    "BM25": RESULTS_BM25[COLS].mean(),
    "Dense (MiniLM)": RESULTS_DENSE[COLS].mean(),
}).T.round(3)
COMPARE


,recall,precision,mrr,ndcg
Random,0.042,0.008,0.010,0.010
BM25,0.875,0.183,0.718,0.680
Dense (MiniLM),0.819,0.167,0.621,0.613


# Part E — hybrid (RRF)

In [12]:
# ── Hybrid: fuse the two rankings ─────────────────────────────
# Each retriever contributes a deep document list (50 chunks -> documents).
def bm25_docs(query, depth=50):
    return bm25_rank(query, k=depth, n_chunks=depth)

def dense_docs(query, depth=50):
    return dense_rank(query, k=depth, n_chunks=depth)


In [13]:
def rrf_fuse(ranked_lists, K=60):
    """Reciprocal Rank Fusion:  score(doc) = sum  1 / (K + rank), rank starting at 1.

    Why ranks and not scores: BM25 scores run to ~30 while cosine similarities sit
    in [0, 1]. Adding them — or normalising them against different distributions —
    invents a weighting. Ranks are comparable by construction, which is why RRF
    needs no score calibration and is the default fusion in practice.

    A document appearing in BOTH lists accumulates two contributions, so agreement
    between retrievers is rewarded. (That property also has a cost: a document only
    one retriever finds can be pushed out of a short context window — measured in
    notebook 02's context builder and written up in the README.)
    """
    scores = {}
    for ranked_list in ranked_lists:
        for rank, doc_id in enumerate(ranked_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (K + rank)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in sorted_scores]


def hybrid_rank(query, k=10, depth=50):
    """Fuse BM25 and dense at the DOCUMENT level, then trim to k."""
    lists = [
        bm25_docs(query, depth),
        dense_docs(query, depth),
    ]

    return rrf_fuse(lists, K=60)[:k]


RESULTS_HYBRID = evaluate(hybrid_rank, k=10)
report(RESULTS_HYBRID, "HYBRID (BM25 + Dense, RRF)")



HYBRID (BM25 + Dense, RRF)   (k=10)

-- mean by query class --
             recall  precision    mrr   ndcg
query_class                                 
literal       1.000      0.150  0.786  0.821
multi-doc     1.000      0.275  1.000  0.877
paraphrase    0.875      0.150  0.667  0.591

-- overall --
recall       0.958
precision    0.192
mrr          0.817
ndcg         0.763

-- per query --
query_id query_class  recall  precision   mrr  ndcg
    q001     literal     1.0        0.2 1.000 0.950
    q002     literal     1.0        0.2 1.000 1.000
    q003     literal     1.0        0.1 0.143 0.333
    q004     literal     1.0        0.1 1.000 1.000
    q005  paraphrase     0.5        0.1 1.000 0.380
    q006  paraphrase     1.0        0.1 0.167 0.356
    q007  paraphrase     1.0        0.2 1.000 1.000
    q008  paraphrase     1.0        0.2 0.500 0.627
    q009   multi-doc     1.0        0.3 1.000 0.852
    q010   multi-doc     1.0        0.2 1.000 0.920
    q011   multi-doc     1.0   

In [14]:
# ── Sensitivity of the RRF constant ───────────────────────────
# K is a hyperparameter, so it gets measured rather than assumed. These are also
# the k=10 numbers the README quotes when it declines to tune K on 12 queries.
for K in (10, 30, 60, 100):
    fn = lambda q, k, K=K: rrf_fuse([bm25_docs(q, 50), dense_docs(q, 50)], K=K)[:k]
    m = evaluate(fn, k=10)[COLS].mean().round(3)
    print(f"RRF K={K:3d} -> recall {m['recall']:.3f} · MRR {m['mrr']:.3f} · nDCG {m['ndcg']:.3f}")


RRF K= 10 -> recall 0.958 · MRR 0.829 · nDCG 0.770


RRF K= 30 -> recall 0.958 · MRR 0.817 · nDCG 0.762


RRF K= 60 -> recall 0.958 · MRR 0.817 · nDCG 0.763


RRF K=100 -> recall 0.958 · MRR 0.817 · nDCG 0.763
